## Setup & Konfigurasi

In [ ]:
# -- Imports -----------------------------------------------------------------
import os
from pathlib import Path

import duckdb
import folium
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from shapely.ops import unary_union
from shapely import prepare

import osmnx as ox

from ODanalysisOfRRU.shared import get_con, haversine_m

# -- Konfigurasi --------------------------------------------------------------
INPUT_PARQUET  = "../DataGPS_parquet/all_gps_data_no_dup.parquet"
OUTPUT_PARQUET = "./processed-data/gps_ring_road_utara_200.parquet"
CACHE_DIR      = "./cache"
BUFFER_METERS  = 200
BATCH_SIZE     = 500_000

os.makedirs(CACHE_DIR, exist_ok=True)

# -- DuckDB Connection --------------------------------------------------------
con = get_con(os.getcwd())
con.execute(f"CREATE OR REPLACE VIEW gps AS SELECT * FROM read_parquet('{INPUT_PARQUET}')")
print(f"✅ View 'gps' dibuat dari: {INPUT_PARQUET}")

✅ View 'gps' dibuat dari: ../DataGPS_parquet/all_gps_data_no_dup.parquet


## Ambil Geometri Ring Road Utara dari OSMnx

In [2]:
# -- Helper Functions ---------------------------------------------------------
TARGET_ROAD_NAMES = ["ring road utara", "siliwangi", "padjajaran", "pajajaran"]

def match_ring_road_name(name_val):
    if isinstance(name_val, str):
        lower = name_val.lower()
        return any(t in lower for t in TARGET_ROAD_NAMES)
    if isinstance(name_val, list):
        return any(match_ring_road_name(n) for n in name_val)
    return False


def load_ring_road_edges(cache_path, place_name="Sleman, Daerah Istimewa Yogyakarta, Indonesia"):
    if os.path.exists(cache_path):
        print(f"📂 Memuat dari cache: {cache_path}")
        return gpd.read_file(cache_path)

    print("🌐 Mengunduh dari OSMnx...")
    G = ox.graph_from_place(place_name, network_type="drive")
    print(f"   Graph: {len(G.nodes)} nodes, {len(G.edges)} edges")
    _, edges = ox.graph_to_gdfs(G)
    ring_edges = edges[edges["name"].apply(match_ring_road_name)].copy()
    ring_edges.to_file(cache_path, driver="GeoJSON")
    print(f"   💾 Disimpan ke cache: {cache_path}")
    return ring_edges


def filter_ring_road_north(edges, lat_threshold=-7.77, lon_threshold=110.342):
    mask = (
        edges["highway"].apply(lambda x: "trunk" in str(x)) &
        edges.geometry.apply(lambda g: g.bounds[1] > lat_threshold and g.bounds[0] > lon_threshold)
    )
    return edges[mask].copy()


# -- Load & Filter Ring Road ---------------------------------------------------
RING_ROAD_CACHE_FULL  = os.path.join(CACHE_DIR, "ring_road_utara_edges.geojson")
RING_ROAD_CACHE_NORTH = os.path.join(CACHE_DIR, "ring_road_utara_filtered.geojson")

all_ring_edges = load_ring_road_edges(RING_ROAD_CACHE_FULL)
ring_road_edges = filter_ring_road_north(all_ring_edges)

print(f"\n📊 Segmen Ring Road (trunk only): {len(ring_road_edges)}")

from collections import Counter
name_counts = Counter()
for val in ring_road_edges["name"].dropna():
    name_counts[str(val)] += 1
print("   Nama jalan:")
for n, c in name_counts.most_common():
    print(f"     • {n}: {c} segmen")

ring_road_edges.to_file(RING_ROAD_CACHE_NORTH, driver="GeoJSON")
print(f"\n💾 Cache disimpan ke: {RING_ROAD_CACHE_NORTH}")

📂 Memuat dari cache: ./cache/ring_road_utara_edges.geojson

📊 Segmen Ring Road (trunk only): 98
   Nama jalan:
     • ['Jalan Padjajaran']: 66 segmen
     • ['Jalan Siliwangi']: 24 segmen
     • ['Jalan Ring Road Utara']: 7 segmen
     • ['Jalan Siliwangi' 'Jalan Padjajaran']: 1 segmen

💾 Cache disimpan ke: ./cache/ring_road_utara_filtered.geojson


## Buat Buffer di Sekitar Ring Road Utara

In [3]:
# -- Buat Buffer Geometri ------------------------------------------------------
ring_road_utm  = ring_road_edges.to_crs(epsg=32749)  # UTM 49S untuk Yogyakarta
road_union     = unary_union(ring_road_utm.geometry)
buffer_utm     = road_union.buffer(BUFFER_METERS)
buffer_gdf     = gpd.GeoDataFrame(geometry=[buffer_utm], crs="EPSG:32749").to_crs(epsg=4326)
buffer_polygon = buffer_gdf.geometry.iloc[0]
prepared_buf   = prepare(buffer_polygon)

# Bounding box untuk pre-filter DuckDB
LON_MIN_BB, LAT_MIN_BB, LON_MAX_BB, LAT_MAX_BB = buffer_polygon.bounds
buffer_centroid = buffer_polygon.centroid

print(f"✅ Buffer {BUFFER_METERS}m dari Ring Road Utara (trunk only)")
print(f"   Bounding Box: lat [{LAT_MIN_BB:.6f}, {LAT_MAX_BB:.6f}], lon [{LON_MIN_BB:.6f}, {LON_MAX_BB:.6f}]")
print(f"   Luas buffer: {buffer_gdf.to_crs(epsg=32749).area.iloc[0] / 1e6:,.2f} km²")

✅ Buffer 200m dari Ring Road Utara (trunk only)
   Bounding Box: lat [-7.771411, -7.742002], lon [110.340299, 110.433240]
   Luas buffer: 4.60 km²


## Visualisasi Ring Road Utara (Folium)

In [4]:
# -- Visualisasi Ring Road Utara -----------------------------------------------
m = folium.Map(location=[buffer_centroid.y, buffer_centroid.x], zoom_start=14, tiles="OpenStreetMap")

folium.GeoJson(
    buffer_gdf[["geometry"]].to_json(),
    name=f"Buffer {BUFFER_METERS}m",
    style_function=lambda _: {
        "fillColor": "#3498db", "color": "#3498db", "weight": 1, "fillOpacity": 0.15,
    },
).add_to(m)

folium.GeoJson(
    ring_road_edges[["geometry"]].to_json(),
    name="Ring Road Utara (trunk)",
    style_function=lambda _: {"color": "#e74c3c", "weight": 4, "opacity": 0.9},
).add_to(m)

folium.LayerControl().add_to(m)
m

## Pre-Filter dengan DuckDB (Bounding Box)

In [5]:
# -- Pre-Filter DuckDB --------------------------------------------------------
total_original = con.execute("SELECT COUNT(*) FROM gps").fetchone()[0]
count_bb = con.execute(f"""
    SELECT COUNT(*) FROM gps
    WHERE latitude  BETWEEN {LAT_MIN_BB} AND {LAT_MAX_BB}
      AND longitude BETWEEN {LON_MIN_BB} AND {LON_MAX_BB}
""").fetchone()[0]

print(f"📊 Total baris asli: {total_original:,}")
print(f"📦 Dalam bounding box: {count_bb:,} ({count_bb/total_original*100:.2f}%)")
print(f"   → Reduksi: {(1 - count_bb/total_original)*100:.2f}%")

📊 Total baris asli: 169,509,324
📦 Dalam bounding box: 21,736,848 (12.82%)
   → Reduksi: 87.18%


## Filter Spasial Presisi + Deduplikasi

In [8]:
# -- Filter Spasial (Batched) --------------------------------------------------
def filter_points_in_buffer(batch_df, polygon):
    """Return boolean mask for points within the buffer polygon."""
    points = [
        Point(lon, lat)
        for lon, lat in zip(batch_df["longitude"], batch_df["latitude"])
    ]
    return pd.Series([polygon.contains(p) for p in points], index=batch_df.index)


def batch_spatial_filter(con, bbox, batch_size, polygon):
    """Iterate over GPS data in batches, returning only points inside buffer polygon."""
    lon_min, lat_min, lon_max, lat_max = bbox
    offset, total_kept, filtered = 0, 0, []

    while True:
        batch = con.execute(f"""
            SELECT maid, latitude, longitude, timestamp
            FROM gps
            WHERE latitude  BETWEEN {lat_min} AND {lat_max}
              AND longitude BETWEEN {lon_min} AND {lon_max}
            LIMIT {batch_size} OFFSET {offset}
        """).df()

        if len(batch) == 0:
            break

        mask = filter_points_in_buffer(batch, polygon)
        kept = int(mask.sum())
        total_kept += kept
        if kept > 0:
            filtered.append(batch.loc[mask].copy())

        print(f"   Batch {len(filtered):>3} | rows: {len(batch):>8,} | kept: {kept:>8,} | total: {total_kept:>10,}")
        offset += batch_size
        if len(batch) < batch_size:
            break

    return filtered, total_kept


print(f"🔄 Filter spasial (batch={BATCH_SIZE:,}, buffer={BUFFER_METERS}m)...")
filtered_chunks, total_kept = batch_spatial_filter(
    con, (LON_MIN_BB, LAT_MIN_BB, LON_MAX_BB, LAT_MAX_BB), BATCH_SIZE, buffer_polygon
)
print(f"\n✅ Selesai! Total titik dalam buffer: {total_kept:,}")

🔄 Filter spasial (batch=500,000, buffer=200m)...
   Batch   1 | rows:  500,000 | kept:   96,654 | total:     96,654
   Batch   2 | rows:  500,000 | kept:   68,915 | total:    165,569
   Batch   3 | rows:  500,000 | kept:   76,092 | total:    241,661
   Batch   4 | rows:  500,000 | kept:   68,568 | total:    310,229
   Batch   5 | rows:  500,000 | kept:   59,595 | total:    369,824
   Batch   6 | rows:  500,000 | kept:   66,138 | total:    435,962
   Batch   7 | rows:  500,000 | kept:   66,711 | total:    502,673
   Batch   8 | rows:  500,000 | kept:   79,950 | total:    582,623
   Batch   9 | rows:  500,000 | kept:   67,836 | total:    650,459
   Batch  10 | rows:  500,000 | kept:   67,992 | total:    718,451
   Batch  11 | rows:  500,000 | kept:  145,915 | total:    864,366
   Batch  12 | rows:  500,000 | kept:   75,291 | total:    939,657
   Batch  13 | rows:  500,000 | kept:   80,352 | total:  1,020,009
   Batch  14 | rows:  500,000 | kept:   65,072 | total:  1,085,081
   Batch  15 

## Gabungkan, Deduplikasi & Periksa Hasil

In [9]:
# -- Deduplikasi & Ringkasan --------------------------------------------------
df_result = pd.concat(filtered_chunks, ignore_index=True) if filtered_chunks else pd.DataFrame(columns=["maid", "latitude", "longitude", "timestamp"])

n_before = len(df_result)
df_result = df_result.drop_duplicates(subset=["maid", "latitude", "longitude", "timestamp"], keep="first").reset_index(drop=True)
n_after  = len(df_result)
n_dupes  = n_before - n_after

print("📊 Ringkasan Filter:")
print(f"   Total baris asli      : {total_original:>14,}")
print(f"   Setelah bounding box  : {count_bb:>14,} ({count_bb/total_original*100:.2f}%)")
print(f"   Setelah filter spasial : {n_before:>14,} ({n_before/total_original*100:.4f}%)")
print(f"   Duplikat dihapus      : {n_dupes:>14,}")
print(f"   ✅ Hasil akhir        : {n_after:>14,} ({n_after/total_original*100:.4f}%)")
print(f"   MAID unik             : {df_result['maid'].nunique():>14,}")
print("\nPreview:")
df_result.head(10)

📊 Ringkasan Filter:
   Total baris asli      :    169,509,324
   Setelah bounding box  :     21,736,848 (12.82%)
   Setelah filter spasial :      3,234,420 (1.9081%)
   Duplikat dihapus      :         44,392
   ✅ Hasil akhir        :      3,190,028 (1.8819%)
   MAID unik             :        177,900

Preview:


,maid,latitude,longitude,timestamp
0,0054d7a0-523c-402f-9ea7-e9b1d4db5736,-7.764816,110.424927,1636689274
1,0054d7a0-523c-402f-9ea7-e9b1d4db5736,-7.764816,110.424927,1636689324
2,0054d7a0-523c-402f-9ea7-e9b1d4db5736,-7.764830,110.424942,1636689731
3,0054d7a0-523c-402f-9ea7-e9b1d4db5736,-7.764840,110.424927,1636689731
4,0054d7a0-523c-402f-9ea7-e9b1d4db5736,-7.764840,110.424919,1636689731
5,0054d7a0-523c-402f-9ea7-e9b1d4db5736,-7.764820,110.424927,1636689731
6,0054d7a0-523c-402f-9ea7-e9b1d4db5736,-7.764816,110.424927,1636689731
7,0054d7a0-523c-402f-9ea7-e9b1d4db5736,-7.764830,110.424911,1636689742
8,0054d7a0-523c-402f-9ea7-e9b1d4db5736,-7.764840,110.424927,1636689742
9,0054d7a0-523c-402f-9ea7-e9b1d4db5736,-7.764830,110.424942,1636689742


In [10]:
# -- Statistik Deskriptif -----------------------------------------------------
print("📈 Statistik:")
print(df_result.describe())

📈 Statistik:
           latitude     longitude     timestamp
count  3.190028e+06  3.190028e+06  3.190028e+06
mean  -7.756913e+00  1.103907e+02  1.643919e+09
std    6.478213e-03  2.578248e-02  6.472079e+06
min   -7.771405e+00  1.103403e+02  1.634947e+09
25%   -7.762739e+00  1.103715e+02  1.637844e+09
50%   -7.759294e+00  1.103938e+02  1.643439e+09
75%   -7.751020e+00  1.104143e+02  1.650135e+09
max   -7.742050e+00  1.104332e+02  1.654595e+09


## Visualisasi Hasil Filter (Folium)

In [11]:
# -- Visualisasi Titik GPS ----------------------------------------------------
m2 = folium.Map(location=[buffer_centroid.y, buffer_centroid.x], zoom_start=14, tiles="OpenStreetMap")

folium.GeoJson(
    buffer_gdf[["geometry"]].to_json(),
    name=f"Buffer {BUFFER_METERS}m",
    style_function=lambda _: {"fillColor": "#3498db", "color": "#3498db", "weight": 1, "fillOpacity": 0.1},
).add_to(m2)

folium.GeoJson(
    ring_road_edges[["geometry"]].to_json(),
    name="Ring Road Utara (trunk)",
    style_function=lambda _: {"color": "#e74c3c", "weight": 3, "opacity": 0.9},
).add_to(m2)

sample_size = min(5_000, len(df_result))
sample = df_result.sample(n=sample_size, random_state=42)
gps_group = folium.FeatureGroup(name=f"GPS Points (sample {sample_size:,})")
for _, row in sample.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=2, color="#2ecc71", fill=True, fill_color="#2ecc71", fill_opacity=0.5, weight=1,
    ).add_to(gps_group)
gps_group.add_to(m2)

folium.LayerControl().add_to(m2)
print(f"📊 Total: {len(df_result):,} titik, {df_result['maid'].nunique():,} MAID unik")
m2

📊 Total: 3,190,028 titik, 177,900 MAID unik


## Simpan ke Parquet Baru

In [12]:
# -- Simpan ke Parquet ---------------------------------------------------------
os.makedirs(os.path.dirname(OUTPUT_PARQUET), exist_ok=True)

con.execute(f"""
    COPY (
        SELECT DISTINCT maid, latitude, longitude, timestamp
        FROM df_result
        ORDER BY maid, timestamp
    ) TO '{OUTPUT_PARQUET}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# Verifikasi output
verify_count   = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PARQUET}')").fetchone()[0]
verify_distinct = con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT maid, latitude, longitude, timestamp
        FROM read_parquet('{OUTPUT_PARQUET}')
    )
""").fetchone()[0]

out_path = Path(OUTPUT_PARQUET)
print(f"✅ Data disimpan!")
print(f"   📁 {out_path.resolve()}")
print(f"   📏 {out_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"   📊 {verify_count:,} baris | distinct: {verify_distinct:,}")
print(f"   🔽 Reduksi: {(1 - verify_count/total_original)*100:.2f}% dari dataset asli")
print(f"   ✅ Duplikat: {'TIDAK ADA ✓' if verify_count == verify_distinct else '⚠️ ADA!'}")

✅ Data disimpan!
   📁 /home/repal/Github/skripsi/ODanalysisOfRRU/processed-data/gps_ring_road_utara_200.parquet
   📏 22.48 MB
   📊 3,190,028 baris | distinct: 3,190,028
   🔽 Reduksi: 98.12% dari dataset asli
   ✅ Duplikat: TIDAK ADA ✓


## Verifikasi Data Output

In [13]:
# -- Verifikasi Output ---------------------------------------------------------
con.execute(f"CREATE OR REPLACE VIEW gps_rru AS SELECT * FROM read_parquet('{OUTPUT_PARQUET}')")

print("=" * 60)
print("VERIFIKASI OUTPUT")
print("=" * 60)

schema = con.execute("""
    SELECT column_name AS "Kolom", data_type AS "Tipe"
    FROM information_schema.columns
    WHERE table_name = 'gps_rru'
    ORDER BY ordinal_position
""").df()
print("\n📋 Schema:")
print(schema.to_string(index=False))

dims = con.execute("""
    SELECT
        COUNT(*)             AS total_rows,
        COUNT(DISTINCT maid) AS unique_maids,
        MIN(latitude)        AS lat_min, MAX(latitude)        AS lat_max,
        MIN(longitude)       AS lon_min, MAX(longitude)       AS lon_max,
        MIN(timestamp)       AS ts_min, MAX(timestamp)       AS ts_max
    FROM gps_rru
""").df()

print(f"\n📊 Dimensi:")
print(f"   Total baris: {int(dims['total_rows'].iloc[0]):,}")
print(f"   MAID unik  : {int(dims['unique_maids'].iloc[0]):,}")
print(f"   Latitude  : {dims['lat_min'].iloc[0]:.6f} → {dims['lat_max'].iloc[0]:.6f}")
print(f"   Longitude : {dims['lon_min'].iloc[0]:.6f} → {dims['lon_max'].iloc[0]:.6f}")

import datetime
ts_min = datetime.datetime.fromtimestamp(int(dims['ts_min'].iloc[0]))
ts_max = datetime.datetime.fromtimestamp(int(dims['ts_max'].iloc[0]))
print(f"   Timestamp : {ts_min} → {ts_max}")

print("\n📝 Preview (5 baris):")
con.execute("SELECT * FROM gps_rru LIMIT 5").df()

VERIFIKASI OUTPUT

📋 Schema:
    Kolom    Tipe
     maid VARCHAR
 latitude  DOUBLE
longitude  DOUBLE
timestamp  BIGINT

📊 Dimensi:
   Total baris: 3,190,028
   MAID unik  : 177,900
   Latitude  : -7.771405 → -7.742050
   Longitude : 110.340340 → 110.433205
   Timestamp : 2021-10-23 07:00:00 → 2022-06-07 16:46:55

📝 Preview (5 baris):


,maid,latitude,longitude,timestamp
0,00000000-0000-0000-0000-000000000000,-7.759640,110.399150,1653797201
1,00005de9-5de9-0c61-5cab-f4e361b157f2,-7.762487,110.415855,1652972624
2,00006d7b-87de-4a2d-a4c8-a3c4d8be3693,-7.762918,110.416820,1637308310
3,00006d7b-87de-4a2d-a4c8-a3c4d8be3693,-7.762918,110.416817,1637308310
4,00006d7b-87de-4a2d-a4c8-a3c4d8be3693,-7.763764,110.416374,1640548269


In [14]:
# -- Cleanup ------------------------------------------------------------------
con.close()
print(f"🏁 Selesai! Output: {Path(OUTPUT_PARQUET).resolve()}")

🏁 Selesai! Output: /home/repal/Github/skripsi/ODanalysisOfRRU/processed-data/gps_ring_road_utara_200.parquet
